## Task 3: Forecast Future Market Trends

### 0. Load Data, Trained Model, and Scaler

We reload the cleaned TSLA data, the trained LSTM model, and the fitted
scaler from Task 2, avoiding the need to retrain. LSTM was selected as the
best-performing model in Task 2 (MAPE of 4.18% vs. ARIMA's 17.24%), so it
is the basis for the future forecast here.

In [6]:
try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import joblib
    from tensorflow.keras.models import load_model

    combined_df = pd.read_csv('../data/processed/combined_assets.csv', parse_dates=['Date'])
    tsla_df = combined_df[combined_df['Ticker'] == 'TSLA'].sort_values('Date').reset_index(drop=True)

    lstm_model = load_model('../data/processed/models/lstm_tsla.keras')
    scaler = joblib.load('../data/processed/models/tsla_scaler.pkl')

    window_size = 60

    print(f"Loaded {len(tsla_df)} rows of TSLA data.")
    print(f"Last date in dataset: {tsla_df['Date'].max()}")
except Exception as e:
    print(f"Error loading data, model, or scaler: {e}")

Loaded 2888 rows of TSLA data.
Last date in dataset: 2026-06-29 00:00:00


d:\Repos\portfolio-optimization\.venv\Lib\site-packages\keras\src\saving\saving_lib.py:843: UserWarning: Skipping variable loading for optimizer 'rm_sprop', because it has 12 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


### 1. Re-establish Train/Test Context

We recreate the same chronological split used in Task 2, so the test-set
residuals (actual vs. predicted error) can be used later to build
confidence intervals around the future forecast.

In [7]:
try:
    train_df = tsla_df[tsla_df['Date'] < '2025-01-01'].reset_index(drop=True)
    test_df = tsla_df[tsla_df['Date'] >= '2025-01-01'].reset_index(drop=True)

    full_close = tsla_df['Adj Close'].values.reshape(-1, 1)
    full_scaled = scaler.transform(full_close)

    print(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")
except Exception as e:
    print(f"Error re-establishing train/test split: {e}")

Train: 2516 rows | Test: 372 rows
